In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
from sentence_transformers import SentenceTransformer
import nltk
from nltk.tokenize import sent_tokenize
import json
import os
from tqdm import tqdm

In [2]:
# Load pretrained Sentence Transformer model and tokenizer

# Choosing all-MiniLM-L6-v2 as it is the smallest model that still performs well
# Other option is all-mpnet-base-v2 which performs slightly better but is much larger and slower
model = SentenceTransformer("all-MiniLM-L6-v2")  

tokenizer = nltk.data.load('tokenizers/punkt/english.pickle') # tokenizer for splitting into sentences

# Create sentence embeddings

# Batch encoding

In [5]:
def is_valid_sentence(sentence):
    """
    Check if a sentence is valid (not just a number or numbered list marker)
    Returns True if sentence is valid, False otherwise
    """
    # Strip whitespace
    sentence = sentence.strip()
    
    # Check if sentence is empty
    if not sentence:
        return False
    
    # Remove common markdown formatting characters
    cleaned = sentence.replace('#', '').replace('*', '').strip()
        
    # Check if sentence is just a number followed by a period (like "1.", "2.", "### 8.", "**2.", etc)
    if cleaned.replace('.', '').isdigit():
        return False
        
    # Check if sentence is too short (less than 3 characters)
    if len(cleaned) < 3:
        return False
        
    # Check if sentence starts with a number and period but has no other content
    if cleaned.split('.')[0].isdigit() and len(cleaned.split('.')) <= 2:
        return False
    
    return True

### Model data encoding

In [6]:
# Config

data_paths = {
    "Small": [
        "/Users/maxschaffelder/Desktop/Thesis/data/exp_1/data/small/raw/generation_results_lora_llama_8b_single_v8.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/data/exp_1/data/small/raw/generation_results_lora_llama_8b_multi_v3.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/data/exp_1/data/small/raw/generation_results_lora_llama_8b_human_v3.jsonl",
        "/Users/maxschaffelder/Desktop/Thesis/data/exp_1/data/small/raw/dolly_test_Llama.jsonl"
    ]
}


# Create output directories
output_base_dir = "/Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings"

for size in ["Small"]:
    os.makedirs(os.path.join(output_base_dir, size), exist_ok=True)

sample_size = 7000
np.random.seed(42)  # Set seed for reproducibility

# Process each dataset
for size, file_paths in data_paths.items():
    print(f"Processing {size} models...")
    
    for file_path in tqdm(file_paths):
        # Extract model name from file path
        file_name = os.path.basename(file_path)
        model_name = file_name.split('.')[0]
        
        # Initialize data containers
        all_model_data = []
        all_model_sentences = []
        
        # Load the data
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                response_key_model = "response_model"
                response_model = data[response_key_model]
                all_model_data.append(response_model)
        
        # Tokenize all responses
        for response_model in all_model_data:
            # response_model_sentences = tokenizer.tokenize(response_model)
            response_model_sentences = sent_tokenize(response_model)
            valid_sentences = [s for s in response_model_sentences if is_valid_sentence(s)]
            all_model_sentences.extend(valid_sentences)
            
            # all_model_sentences.extend(response_model_sentences)
        
        # Sample sentences
        if len(all_model_sentences) >= sample_size:
            random_indices_model = np.random.choice(len(all_model_sentences), size=sample_size, replace=False)
            sampled_model_sentences = [all_model_sentences[i] for i in random_indices_model]
        else:
            print(f"Warning: Not enough model sentences in {file_name}. Using all available.")
            sampled_model_sentences = all_model_sentences
        
        print(f"  {file_name}: {len(sampled_model_sentences)} model sentences")
        
        # Calculate embeddings
        print(f"  Calculating embeddings for {file_name}...")
        embeddings_model = model.encode(sampled_model_sentences)
        
        # Save model embeddings
        model_output_file = os.path.join(output_base_dir, size, f"{model_name}_embeddings.jsonl")
        with open(model_output_file, 'w') as f:
            for sentence, embedding in zip(sampled_model_sentences, embeddings_model):
                entry = {
                    "sentence": sentence,
                    "embedding": embedding.tolist(),
                    "source": model_name
                }
                f.write(json.dumps(entry) + '\n')
        
        print(f"  Saved embeddings to {model_output_file}")

print("All embeddings have been generated and saved.")



Processing Small models...


  0%|          | 0/4 [00:00<?, ?it/s]

  generation_results_lora_llama_8b_single_v8.jsonl: 7000 model sentences
  Calculating embeddings for generation_results_lora_llama_8b_single_v8.jsonl...


 25%|██▌       | 1/4 [00:12<00:37, 12.56s/it]

  Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings/Small/generation_results_lora_llama_8b_single_v8_embeddings.jsonl
  generation_results_lora_llama_8b_multi_v3.jsonl: 7000 model sentences
  Calculating embeddings for generation_results_lora_llama_8b_multi_v3.jsonl...


...

  Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings/Small/generation_results_lora_llama_8b_multi_v3_embeddings.jsonl
  generation_results_lora_llama_8b_human_v3.jsonl: 7000 model sentences
  Calculating embeddings for generation_results_lora_llama_8b_human_v3.jsonl...


 75%|███████▌  | 3/4 [00:32<00:10, 10.47s/it]

  Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings/Small/generation_results_lora_llama_8b_human_v3_embeddings.jsonl
  dolly_test_Llama.jsonl: 7000 model sentences
  Calculating embeddings for dolly_test_Llama.jsonl...


100%|██████████| 4/4 [00:41<00:00, 10.44s/it]

  Saved embeddings to /Users/maxschaffelder/Desktop/Thesis/Data/exp_1/data/sentence_embeddings/Small/dolly_test_Llama_embeddings.jsonl
All embeddings have been generated and saved.
